# 11 — Attention Head Visualization

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/11_attention_visualization.ipynb)

Visualise **self-attention patterns** across DantinoX paradigms.

Key differences:
- **AR (causal)**: strict lower-triangular — each token attends only to past tokens
- **Discrete (bidirectional)**: full matrix — tokens attend everywhere
- **Heads specialise**: some track local neighbours, others span the full sequence

**Sections:** models & reference sequence · extract attention weights · heatmaps · head entropy · layer-depth · side-by-side comparison

**Runtime**: ~10 min · GPU (T4)

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import jax
print('Devices:', jax.devices())

In [ ]:
!pip install -q "dantinox[all] @ git+https://github.com/winstonsmith1897/DantinoX.git"

In [ ]:
import urllib.request, os
if not os.path.exists('tiny_shakespeare.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'tiny_shakespeare.txt')
    print('Downloaded tiny_shakespeare.txt')
else:
    print('tiny_shakespeare.txt already present')

## 1 — Build models and reference sequence

Same architecture; only `causal` differs.

In [ ]:
import dantinox as dx
import jax, jax.numpy as jnp, numpy as np, matplotlib.pyplot as plt
from flax import nnx
from dantinox.core.model  import Transformer
from dantinox.core.config import ModelConfig

DIM, N_HEADS, HEAD_SIZE, N_BLOCKS = 128, 4, 32, 4
SEQ_LEN, VOCAB = 32, 256

cfg_ar   = ModelConfig(dim=DIM,n_heads=N_HEADS,head_size=HEAD_SIZE,num_blocks=N_BLOCKS,
                       vocab_size=VOCAB,max_context=SEQ_LEN+1,causal=True, dropout=0.0)
cfg_disc = ModelConfig(dim=DIM,n_heads=N_HEADS,head_size=HEAD_SIZE,num_blocks=N_BLOCKS,
                       vocab_size=VOCAB,max_context=SEQ_LEN+1,causal=False,dropout=0.0)

ar_model   = Transformer(cfg_ar,   rngs=nnx.Rngs(0))
disc_model = Transformer(cfg_disc, rngs=nnx.Rngs(0))

REF = "To be, or not to be, that is the question:"
x   = jnp.array([[ord(c) % VOCAB for c in REF[:SEQ_LEN]]],dtype=jnp.int32)

def n_params(m): return sum(v.size for v in jax.tree_util.tree_leaves(nnx.state(m,nnx.Param)))
print(f'AR   {n_params(ar_model):,} params')
print(f'Disc {n_params(disc_model):,} params')
print(f'Input: {x.shape}  seq: {REF[:SEQ_LEN]!r}')

## 2 — Extracting attention weights

Tries the native `return_attention_weights=True` flag first (DantinoX ≥ 0.4).
Falls back to computing Q·Kᵀ scores manually from `model.blocks[i].attn`.

Result shape: `[n_layers, n_heads, T, T]`.

In [ ]:
def get_attn(model, x, causal, cfg):
    # Native flag ──────────────────────────────────────────────────────────────
    try:
        out = model(x, deterministic=True, return_attention_weights=True)
        if hasattr(out,'attentions') and out.attentions is not None:
            return np.array(out.attentions[0])
    except TypeError:
        pass

    # Fallback: manual Q·Kᵀ ──────────────────────────────────────────────────
    B, T   = x.shape
    H, HS  = cfg.n_heads, cfg.head_size
    attn_all = []

    # Attempt to grab token embeddings (attribute name may vary)
    embed_W = None
    for attr in ('embed_tokens','token_embed','embedding'):
        sub = getattr(model, attr, None)
        if sub is not None:
            vals = list(jax.tree_util.tree_leaves(nnx.state(sub, nnx.Param)))
            if vals:
                embed_W = vals[0].value if hasattr(vals[0],'value') else vals[0]
                break
    if embed_W is None:
        embed_W = jnp.zeros((VOCAB, DIM))

    h = embed_W[x[0]][None]   # [1, T, D]

    for block in model.blocks:
        try:
            q = block.attn.q_proj(h)
            k = block.attn.k_proj(h)
        except AttributeError:
            attn_all.append(np.ones((H,T,T))/T)
            h = block(h, deterministic=True); continue

        q = q.reshape(1,T,H,HS).transpose(0,2,1,3)
        k = k.reshape(1,T,H,HS).transpose(0,2,1,3)
        scores = (q @ k.transpose(0,1,3,2)) / jnp.sqrt(float(HS))
        if causal:
            scores = jnp.where(jnp.tril(jnp.ones((T,T),bool)), scores, -1e9)
        attn_all.append(np.array(jax.nn.softmax(scores,-1)[0]))
        h = block(h, deterministic=True)

    return np.stack(attn_all)   # [L, H, T, T]

print('Extracting AR attention...')
ar_attn   = get_attn(ar_model,   x, causal=True,  cfg=cfg_ar)
print(f'  {ar_attn.shape}')
print('Extracting Discrete attention...')
disc_attn = get_attn(disc_model, x, causal=False, cfg=cfg_disc)
print(f'  {disc_attn.shape}')

## 3 — AR causal attention heatmaps

Strict lower-triangular in every head. Heads differ in which prior tokens they weight.

In [ ]:
def plot_heads(attn_layer, title, cmap='Blues'):
    H  = attn_layer.shape[0]
    nc = min(H,4); nr = (H+nc-1)//nc
    fig, axes = plt.subplots(nr,nc,figsize=(nc*3.2,nr*3.0))
    axes = np.array(axes).ravel()
    for h in range(H):
        im = axes[h].imshow(attn_layer[h],cmap=cmap,aspect='auto',vmin=0)
        axes[h].set_title(f'Head {h}',fontsize=9)
        axes[h].set_xlabel('Key',fontsize=7); axes[h].set_ylabel('Query',fontsize=7)
        plt.colorbar(im,ax=axes[h],fraction=0.046,pad=0.04)
    for h in range(H,len(axes)): axes[h].set_visible(False)
    fig.suptitle(title,fontsize=10,fontweight='bold')
    plt.tight_layout(); plt.show()

plot_heads(ar_attn[0],  f'AR — Layer 0  ({N_HEADS} heads)','Blues')
plot_heads(ar_attn[-1], f'AR — Layer {N_BLOCKS-1} ({N_HEADS} heads)','Blues')

## 4 — Discrete bidirectional attention heatmaps

No causal mask: full attention matrix. All positions decoded in parallel.

In [ ]:
plot_heads(disc_attn[0],  f'Discrete — Layer 0  ({N_HEADS} heads)','Reds')
plot_heads(disc_attn[-1], f'Discrete — Layer {N_BLOCKS-1} ({N_HEADS} heads)','Reds')

## 5 — Head entropy: focused vs. diffuse

H = −∑ p·log(p). High entropy = diffuse/global; low = focused/local.

In [ ]:
def head_entropy(attn):
    p = np.clip(attn,1e-9,1.0)
    return -(p*np.log(p)).sum(-1).mean(-1)   # [L, H]

ar_He, disc_He = head_entropy(ar_attn), head_entropy(disc_attn)

fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax,He,title,cmap in [(axes[0],ar_He,'AR — head entropy (nats)','Blues'),
                          (axes[1],disc_He,'Discrete — head entropy (nats)','Reds')]:
    im = ax.imshow(He,cmap=cmap,aspect='auto')
    ax.set_xlabel('Head'); ax.set_ylabel('Layer')
    ax.set_xticks(range(N_HEADS)); ax.set_yticks(range(N_BLOCKS))
    ax.set_title(title,fontweight='bold')
    plt.colorbar(im,ax=ax,label='Entropy (nats)')
plt.suptitle('High entropy = diffuse  ·  Low entropy = focused',y=1.02)
plt.tight_layout(); plt.show()

## 6 — Layer-depth: mean attention distance

Expected |query−key| position gap. Early layers attend locally; later layers reach further.

In [ ]:
def mean_attn_dist(attn):
    T    = attn.shape[-1]
    pos  = np.arange(T)
    dist = np.abs(pos[:,None]-pos[None,:])
    return (attn*dist[None,None]).sum(-1).mean(-1)   # [L, H]

ar_dist   = mean_attn_dist(ar_attn)
disc_dist = mean_attn_dist(disc_attn)
layers    = np.arange(N_BLOCKS)

fig, ax = plt.subplots(figsize=(8,4))
for dist,lbl,col in [(ar_dist,'AR','#1f77b4'),(disc_dist,'Discrete','#d62728')]:
    mu,sig = dist.mean(1),dist.std(1)
    ax.plot(layers,mu,marker='o',lw=2,color=col,label=lbl)
    ax.fill_between(layers,mu-sig,mu+sig,alpha=0.15,color=col)
ax.set_xlabel('Layer'); ax.set_ylabel('Mean attention distance (tokens)')
ax.set_title('Attention distance per layer  (±std over heads)',fontweight='bold')
ax.set_xticks(layers); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7 — Side-by-side: all layers, head 0

The filled upper triangle is the defining signature of bidirectional attention.

In [ ]:
fig, axes = plt.subplots(2,N_BLOCKS,figsize=(N_BLOCKS*3.4,7))
for li in range(N_BLOCKS):
    for ri,(attn,lbl,cmap) in enumerate([(ar_attn,'AR','Blues'),(disc_attn,'Discrete','Reds')]):
        ax = axes[ri,li]
        ax.imshow(attn[li,0],cmap=cmap,aspect='auto',vmin=0)
        ax.set_title(f'{lbl}  L{li} H0',fontsize=8,fontweight='bold')
        if li==0: ax.set_ylabel('Query',fontsize=8)
        ax.set_xlabel('Key',fontsize=8); ax.tick_params(labelsize=6)
fig.suptitle(
    f'AR (causal, lower-tri) vs Discrete (bidirectional, full)\n'
    f'All {N_BLOCKS} layers · head 0 · same reference sequence',
    fontsize=10,fontweight='bold')
plt.tight_layout(); plt.show()